# Example: Human Protein Atlas — Subcellular Phenotypes

Analyze large-scale subcellular protein localization patterns using high-resolution microscopy images from the Human Protein Atlas (HPA).

**Dataset:** [Human Protein Atlas](https://www.proteinatlas.org/) (High-resolution human protein imagery)

**Key Steps:**
1. **Metadata Linking:** Link master CSV metadata (protein locations, cell types) to discovered image files.
2. **Feature Extraction:** Compute high-resolution DINOv2 embeddings and basic intensity/morphology properties.
3. **Analysis & Exploration:** Cluster cells by subcellular localization patterns and interactively explore the phenospace in 2D.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to the HPA dataset and the master metadata CSV. This metadata contains critical information about protein locations and cell types.

In [ ]:
DATA_DIR = "/datasets/HPA/512_whole_images"
METADATA_PATH = "/datasets/HPA/Master_fovHPA_512.csv"

## 2. Load Metadata & Preprocessing

We load the metadata CSV and extract the filename from the path to enable matching with the image files on disk.

In [ ]:
import os

df_metadata = pd.read_csv(METADATA_PATH)

# Select relevant columns and extract filename
df_metadata = df_metadata[["Path", "protein_location", "cell_type"]]
df_metadata["filename"] = df_metadata["Path"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])

print(f"Loaded {len(df_metadata)} HPA entries.")
df_metadata.head()

## 3. Data Discovery

The `find_files` method scans the data directory for images and links them with the metadata using the filename column we prepared.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR), 
    metadata_fn=make_dataframe_metadata_fn(df_metadata)
)
file_df.head()

## 4. Feature Extraction

We extract high-resolution DINOv2 embeddings to capture complex subcellular patterns. Additionally, we compute basic morphological and intensity properties to complement the deep features.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=32,
    resize_size=518,
    channel_mode="split",
)

df_properties = pheno.compute_properties(
    property_preset="basic",
    n_jobs=8,
)

## 5. Visualization

Visualize the high-dimensional embeddings in a 2D space using t-SNE, colored by cell type to observe how different localizations cluster.

In [ ]:
pheno.plot_tsne(color_by="cell_type")

## 6. Interactive Exploration

Launch the interactive explorer to visually inspect individual cells and their corresponding subcellular localization patterns directly from the plot.

In [ ]:
e = pheno.create_interactive_explorer(hover_features=["cell_type", "protein_location"])